# **(ADD THE NOTEBOOK NAME HERE)**

## Objectives

* Write your notebook objective here, for example, "Fetch data from Kaggle and save as raw data", or "engineer features for modelling"

## Inputs

* Write down which data or information you need to run the notebook 

## Outputs

* Write here which files, code or artefacts you generate by the end of the notebook 

## Additional Comments

* If you have any additional comments that don't fit in the previous bullets, please state them here. 



---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [23]:
import os
current_dir = os.getcwd()
current_dir  

'/Users/apple/Desktop/Project One'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [24]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [25]:
current_dir = os.getcwd()
current_dir

'/Users/apple/Desktop'

# Libraries Import


In [26]:
# Libraries being used in the project
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Section 1

# Initial Data Exploration

This section loads the raw data set and inspects its structure before any changes are made.The 
purpose is to establish a factual baseline — size, column names, data types, and a first look at 
actual values — so that every decision made in the data cleaning section can be traced back to 
something observed here, rather than assumed.

In [27]:
# Loading CSV File 
df = pd.read_csv('Data_Set/Raw_Data/BankChurners 2.csv')
df.head() # Display the first 5 rows of the DataFrame

FileNotFoundError: [Errno 2] No such file or directory: 'Data_Set/Raw_Data/BankChurners 2.csv'

# Data Set Size
The shape of the dataset is checked first, to establish how many customer records and how many 
variables are being worked with.

In [ ]:
df.shape # Display the shape of the DataFrame (rows, columns)

# Column Names 
The full list of column names is reviewed to identify which variables exist and which are likely 
to be relevant to the hypotheses being tested later in the project.

In [ ]:
df.columns # Display the column names of the DataFrame

# Data Types and Non-Null Counts
df.info() is used to check the data type of each column and confirm whether any nulls are 
present. This is also the first opportunity to notice anything unusual — for example, columns 
that are numeric but behave more like categories, or object columns that may later need encoding.

In [ ]:
df.info() # Display a concise summary of the DataFrame, including the number of non-null entries and data types

# Descriptive Stats
Summary statistics are generated for all numeric columns. This gives an initial sense of scale 
and spread — minimum, maximum, mean, and quartiles — for every numeric variable in the dataset, 
ahead of narrowing in on specific columns later.

In [ ]:
df.describe().round(2)  # Display the data types of each column in the DataFrame

# Distinct Values Counts
The number of unique values per column is checked next. This distinguishes genuinely continuous 
numeric columns (many distinct values) from categorical or low-cardinality columns (a handful of 
repeated values), which informs how each column should be treated later — for example, which 
columns are suited to .describe() versus .value_counts().

In [ ]:
df.nunique().sort_values()

---

# Section 2

# Data Cleaning Section 

This section documents every cleaning action taken on the dataset, along with the reasoning 
behind it. Where the data was already found to be clean (for example, no duplicate rows), this is 
stated explicitly rather than omitted, since verifying data quality is itself part of the cleaning 
process and not only the corrections that follow from it.

# Duplicate Records

Two checks are carried out: one for fully duplicated rows across the dataset, and a second 
specifically on the CLIENTNUM column. A duplicate row is a general data quality issue, but a 
duplicate customer ID would indicate the same customer appearing twice, which could distort any 
churn rate calculated later.

In [ ]:

df.duplicated().sum() # Check for fully duplicated rows

In [ ]:

df['CLIENTNUM'].duplicated().sum() # Check for duplicate customer IDs specifically

No duplicate rows or duplicate customer IDs were found. No action was required, and this result 
is recorded as evidence that the check was carried out.

# Checking For Missing Values


df.isnull().sum() was used to confirm any null values.

In [ ]:
df.isnull().sum()

# 'Unknown' Values In Categories Columns

df.info() in Section 1 confirmed there are no literal null values in the dataset. However, three 
categorical columns — Education_Level, Marital_Status, and Income_Category — contain the 
string 'Unknown', which functions as missing data even though it is not registered as NaN by 
pandas. Each column is checked individually to establish how significant this is.

In [ ]:
for col in ['Education_Level', 'Marital_Status', 'Income_Category']:
    unknown_count = (df[col] == 'Unknown').sum()
    unknown_pct = (df[col] == 'Unknown').mean() * 100
    print(f"{col}: {unknown_count} rows ({unknown_pct:.1f}%)")

Between roughly 7% and 15% of rows are affected across these three columns. It was decided that 
these values would be kept as their own category rather than removed or imputed. Removing them 
would discard a meaningful proportion of the dataset and could bias the sample if non-disclosure 
is not random across customers. Imputing a value (such as the most frequent category) would 
introduce a false sense of certainty about information the customer did not actually provide. 
Since none of these three columns are used directly in the hypotheses tested later in this 
project, this decision does not affect the analysis, but it is documented here for completeness.

# Outlier Detection

Outliers are checked using the interquartile range (IQR) method, applied to the numeric columns 
most relevant to the hypotheses under investigation: utilization ratio, revolving balance, credit 
limit, transaction change ratios, and contact frequency.


In [ ]:
def iqr_outlier_count(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower_bound, upper_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((series < lower_bound) | (series > upper_bound)).sum()

outlier_check_cols = ['Avg_Utilization_Ratio', 'Total_Revolving_Bal', 'Credit_Limit',
                       'Total_Amt_Chng_Q4_Q1', 'Total_Ct_Chng_Q4_Q1', 'Contacts_Count_12_mon']

for col in outlier_check_cols:
    print(f"{col}: {iqr_outlier_count(df[col])} potential outliers")

Flagged values were reviewed and kept rather than removed. High values in columns such as 
Credit_Limit or Total_Revolving_Bal reflect genuine high-value customers rather than data entry 
errors. For the hypotheses concerning utilization and limit-maxing behaviour specifically, extreme 
values are part of the pattern being investigated, and removing them would remove the evidence the 
analysis depends on.

# Encoding The Target Variable and Drop Column

The Attrition_Flag column is stored as text (`'Existing Customer'` or `'Attrited Customer'`). A 
binary numeric version is created so that it can be used directly in the statistical tests applied 
to each hypothesis later in the project.

In [ ]:

df_clean = df.drop(columns=['CLIENTNUM']) # Drop the customer ID column, which has no analytical value


df_clean['Churn'] = (df_clean['Attrition_Flag'] == 'Attrited Customer').astype(int) # Encode the target variable

df_clean['Churn'].value_counts(normalize=True).round(3)

Churn
0    0.839
1    0.161
Name: proportion, dtype: float64

The encoded column confirms that approximately 84% of customers are retained and 16% have 
churned. This class imbalance does not require correction for the descriptive and statistical work 
carried out in this project, but it is noted here as it should be considered when interpreting 
churn rate differences between groups in later sections.

# Feature Engineering: Amount-Count Mismatch Ratio

A new column is engineered here to support Hypothesis 2: the ratio of the change in transaction 
amount to the change in transaction count between Q4 and Q1 (Total_Amt_Chng_Q4_Q1 divided by 
Total_Ct_Chng_Q4_Q1). A ratio far from 1 indicates that a customer's spending value and spending 
frequency moved out of step with one another - for example, a handful of unusually large 
transactions rather than steady regular activity. This column does not exist in the raw dataset 
and is created here, as part of the Transform stage, so that it is available in the cleaned 
dataset used for all analysis going forward.

In [ ]:

df_clean['Amt_Count_Mismatch_Ratio'] = df_clean['Total_Amt_Chng_Q4_Q1'] / df_clean['Total_Ct_Chng_Q4_Q1'] # Feature engineering: amount-count mismatch ratio for Hypothesis 2

# Save Clean Data Set

The cleaned dataset is saved as a separate file. This ensures that the visualisation work planned 
for the following stage of the project loads from a single, consistent, and fully documented 
source, rather than repeating the cleaning steps.


---

NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Push files to Repo

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [ ]:
import os
try:
  # create your folder here
  # os.makedirs(name='')
except Exception as e:
  print(e)
